<h1> ALP  Machine Learning 2025 </h1><br>
Kelvin Marcello Pieter - 07060123100xx<br>
Rinaldy Tanriady Tan - 0706012310055<br>
Stefanie Aurelia Mercy Agahari  - 0706012310056<br>


<h1> OverView Data </h1><br>

age = usia individu dalam satuan tahun<br>
workclass = jenis kelas pekerjaan individu (misalnya Private, Self-emp, Government)<br>
fnlwgt = final weight yang digunakan oleh U.S. Census Bureau untuk merepresentasikan populasi<br>
education = tingkat pendidikan terakhir individu<br>
education-num = representasi numerik dari tingkat pendidikan<br>
marital-status = status pernikahan individu<br>
occupation = jenis pekerjaan individu<br>
relationship = hubungan individu dalam rumah tangga<br>
race = ras individu<br>
sex = jenis kelamin individu<br>
capital-gain = keuntungan modal yang diperoleh individu<br>
capital-loss = kerugian modal yang dialami individu<br>
hours-per-week = jumlah jam kerja individu per minggu<br>
native-country = negara asal individu<br>
income = kategori pendapatan tahunan individu (≤50K atau >50K) (target variable)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import pandas as pd
import numpy as np
# Pastikan semua impor dari sklearn.model_selection ada:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
# Pastikan semua impor metrik ada:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

## Load Data

In [ ]:
# Load datasetnya

url = "/Users/stefanieagahari/Downloads/ALP_APP/data/adult.csv"
df = pd.read_csv(url, encoding="latin1")
df.head()

In [ ]:
df.info()

In [ ]:
print(df.shape)

In [ ]:
df.describe()

In [ ]:
# Check for NaN values
df.isna().sum()

In [ ]:
# Check for null values 
df.isnull().sum()

In [ ]:
# cek ada dupli/tidak
df.duplicated().sum()

In [ ]:
for col in df.columns:
    print(f"=== {col} ===")
    print(df[col].value_counts(dropna=False))
    print("\n")

In [ ]:
print(df.columns.tolist())

## Encoding

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean = df_clean.replace('?', np.nan)

In [ ]:
# Check for null values 
df_clean.isnull().sum()

In [ ]:
categorical_cols_with_nan = ['workclass', 'occupation', 'native.country']

for col in categorical_cols_with_nan:
    mode_value = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_value)

In [ ]:
# Check for null values 
df_clean.isnull().sum()

In [ ]:
# Variabel Target (Biner)
df_clean['income'] = df_clean['income'].map({'<=50K': 0, '>50K': 1})

# sex (Biner)
df_clean['sex'] = df_clean['sex'].map({'Male': 0, 'Female': 1}) 

# relationship (Dikelompokkan)
df_clean['relationship_status'] = df_clean['relationship'].apply(lambda x: 1 if x in ['Husband', 'Wife'] else 0)
df_clean = df_clean.drop('relationship', axis=1) # Drop kolom asli

# race (Dikelompokkan untuk Audit Fairness)
df_clean['race_grouped'] = df_clean['race'].apply(lambda x: 1 if x == 'White' else 0)
df_clean = df_clean.drop('race', axis=1) # Drop kolom asli

# marital.status (Dikelompokkan)
df_clean['is_married'] = df_clean['marital.status'].apply(lambda x: 1 if x.startswith('Married') else 0)
df_clean = df_clean.drop('marital.status', axis=1) # Drop kolom asli


# 2.3 One-Hot Encoding (OHE) Selektif untuk Fitur Multi-Kategori Nominal
# Alasan: Mencegah model linear (Logistic Regression) menafsirkan urutan numerik palsu (ordinalitas)
# pada fitur nominal (e.g., occupation). Drop_first=True mengurangi multicollinearity.
ohe_features = ['workclass', 'education', 'occupation', 'native.country']
df_clean = pd.get_dummies(df_clean, columns=ohe_features, drop_first=True)

# Drop 'fnlwgt'
# Alasan: fnlwgt adalah bobot sensus yang tidak relevan untuk prediksi pendapatan individu.
df_clean = df_clean.drop('fnlwgt', axis=1)

In [ ]:
df_clean.head()

In [ ]:
for col in df_clean.columns:
    print(f"=== {col} ===")
    print(df_clean[col].value_counts(dropna=False))
    print("\n")

In [ ]:
X = df_clean.drop('income', axis=1)
y = df_clean['income']

In [ ]:

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical columns:", list(num_cols))
print("Categorical columns:", list(cat_cols))


In [ ]:
# --- 3. Train-Test Split dan Penskalaan (Scaling) ---

# 3.1 Train-Test Split (80:20 Stratified)
# Alasan: test_size=0.2 (20%) sesuai proposal. Stratify=y memastikan proporsi kelas target sama di data latih dan uji, 
# yang krusial karena adanya class imbalance (<=50K jauh lebih banyak).
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42, 
    stratify=y       
)

In [ ]:
# 3.2 Penskalaan (Scaling) Fitur Numerik/Label-Encoded
# Alasan: StandardScaler memastikan semua fitur (terutama untuk Logistic Regression) memiliki rata-rata 0 dan varians 1, 
# sehingga tidak ada fitur yang mendominasi hanya karena skala angkanya besar. Dilakukan setelah split untuk menghindari data leakage.
numerical_features = [
    'age', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week', 
    'sex', 'is_married', 'relationship_status', 'race_grouped'
]
scaler = StandardScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test[numerical_features] = scaler.transform(X_test[numerical_features])

In [ ]:
# --- 4. Pelatihan Model Baseline ---
print("\n4. Evaluasi Model Baseline:")
models_baseline = {
    "Random Forest (Awal)": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Logistic Regression (Baseline)": LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1),
    "Decision Tree (Baseline)": DecisionTreeClassifier(random_state=42)
}
for name, model in models_baseline.items():
    # Model dilatih pada data X_train yang sudah diperbaiki
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    # zero_division=0 untuk menangani warning F1=0, jika terjadi.
    f1 = f1_score(y_test, y_pred, zero_division=0) 
    print(f"   {name} F1-Score Awal: {f1:.4f}")

In [ ]:
# --- 5. Hyperparameter Tuning (Grid Search) pada Random Forest Utama ---
print("\n5. Hyperparameter Tuning Random Forest:")

param_grid = {
    'n_estimators': [50, 100],  
    'max_depth': [10, 20],      
    'min_samples_leaf': [5, 10], 
    'class_weight': [None, 'balanced'] 
}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           scoring='f1', cv=3, verbose=0, n_jobs=-1)

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

print(f"   Parameter Terbaik: {grid_search.best_params_}")

# Evaluasi Model Tuned
y_pred_tuned = best_rf.predict(X_test)
print("\n   Laporan Klasifikasi Random Forest (Tuned):")
print(classification_report(y_test, y_pred_tuned, target_names=['<=50K', '>50K']))

In [ ]:
# --- 6. AUDIT FAIRNESS BERDASARKAN SUBKELOMPOK ---
print("\n6. AUDIT FAIRNESS (Recall, Precision, dan F1 >50K) ---")

def evaluate_subgroup_fairness(model, X_test, y_test, sensitive_column):
    results = {}
    subgroups = X_test[sensitive_column].unique()
    
    for subgroup_value in subgroups:
        idx = X_test[sensitive_column] == subgroup_value
        X_sub = X_test.loc[idx]
        y_sub = y_test.loc[idx]
        y_pred_sub = model.predict(X_sub)
        
        f1 = f1_score(y_sub, y_pred_sub, zero_division=0)
        recall = recall_score(y_sub, y_pred_sub, zero_division=0)
        precision = precision_score(y_sub, y_pred_sub, zero_division=0)
        
        label_map = {
            'sex': {0: 'Male (0)', 1: 'Female (1)'},
            'race_grouped': {0: 'Non-White (0)', 1: 'White (1)'}
        }
        label = label_map[sensitive_column].get(subgroup_value, str(subgroup_value))
            
        results[label] = {
            'N': len(y_sub),
            'Recall (>50K)': recall,
            'Precision (>50K)': precision,
            'F1-Score (>50K)': f1
        }
    
    return pd.DataFrame(results).T

# 6.1 Audit Berdasarkan Jenis Kelamin
fairness_sex = evaluate_subgroup_fairness(best_rf, X_test, y_test, 'sex')
print("\n6.1 Fairness Audit: Jenis Kelamin (sex)")
print(fairness_sex.to_markdown(numalign="left", stralign="left"))

# 6.2 Audit Berdasarkan Ras
fairness_race = evaluate_subgroup_fairness(best_rf, X_test, y_test, 'race_grouped')
print("\n6.2 Fairness Audit: Ras (race_grouped)")
print(fairness_race.to_markdown(numalign="left", stralign="left"))

### Fairness Audit: Jenis Kelamin (sex)

- Interpretasi: Model menunjukkan disparitas Recall sebesar 10.04% (Recall Male lebih tinggi daripada Female). Artinya, model lebih sering berhasil mengidentifikasi individu Male yang berpendapatan tinggi ($>50$K) dibandingkan individu Female yang berpendapatan tinggi.

- Meskipun Recall untuk Male lebih tinggi, Precision untuk Female ($0.620$) sedikit lebih tinggi daripada Male ($0.565$), menunjukkan bahwa prediksi positif model untuk Female lebih akurat.

### Fairness Audit: Ras (race_grouped)

- Interpretasi: Terdapat disparitas Recall sebesar 5.30% (Recall White lebih tinggi daripada Non-White). Ini menunjukkan bahwa model lebih baik dalam mengidentifikasi individu dari kelompok White yang berpendapatan tinggi.
- Kelompok Non-White memiliki Recall ($0.797$) dan Precision ($0.535$) yang lebih rendah dibandingkan kelompok White, menunjukkan adanya potensi bias terhadap kelompok minoritas.

In [ ]:
# --- 7. ANALISIS FEATURE IMPORTANCE ---
print("\n7. ANALISIS FEATURE IMPORTANCE ---")

r = permutation_importance(
    best_rf, 
    X_test, 
    y_test, 
    n_repeats=10, 
    random_state=42, 
    n_jobs=-1, 
    scoring='f1' 
)

feature_importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance_Mean': r.importances_mean
})

feature_importance_df = feature_importance_df.sort_values(by='Importance_Mean', ascending=False)

print("\n7.1 10 Fitur Terpenting (Permutation Importance):")
print(feature_importance_df.head(10).to_markdown(index=False, numalign="left", stralign="left"))

In [ ]:
# ===============================
# 11. EVALUASI MODEL
# ===============================
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)


In [ ]:
# ===============================
# 12. CONFUSION MATRIX & REPORT
# ===============================
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
